# Experiment: XAUUSD ML + MACD / RSI / SMC

**คำถาม** อินดิเคเตอร์ที่เพิ่มเข้า desk เมื่อ 2026-10-05 ช่วย ML bull/bear ของทองได้ไหม
- MACD แบบ ChrisMoody `CM_MacD_Ult_MTF`: signal = SMA9, histogram 4 สี, MACD cross
- RSI แบบ TradingView: RSI-based MA (SMA14) และ regular divergence (pivot 5/5, 5–60 แท่ง)
- Smart Money Concepts (LuxAlgo port): swing/internal structure, premium/discount, order block, FVG, sweep ของ equal high/low

ทดสอบ 2 แบบ
1. **เป็น feature ของโมเดล**: ML เดิม เทียบกับ ML + อินดิเคเตอร์, ML + SMC และ ML + ทั้งสองอย่าง
2. **เป็นตัวกรองสัญญาณ ML** (ไม่เทรนใหม่): กรอง momentum RSI+MACD, กรองตาม SMC swing trend, ใช้ setup "sweep continuation" ที่ desk ใช้ และ setup เดี่ยวๆ ไม่มี ML

**เกณฑ์ตั้งไว้ก่อนรัน** แบบไหนจะนับว่า "ดีขึ้น" ต้องผ่านทั้งสามข้อ
- R เฉลี่ยต่อไม้สูงกว่า ML เดิม และ bootstrap 90% CI ของผลต่างไม่คร่อมศูนย์
- ไม่แย่ลงทั้งในช่วงที่ไม่เคยเห็น (2025-06..09) และช่วงที่ v0.1 เคยดูแล้ว (2025-10..2026-01)
- มีไม้อย่างน้อย 100 ไม้

ถ้าไม่ผ่าน สรุปว่า "ไม่มีหลักฐาน" ไม่ใช่ "ดีขึ้นนิดหน่อย"

In [1]:
# Setup: imports and reproducibility
from __future__ import annotations

import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

NB_DIR = Path.cwd()
assert (NB_DIR / "xau_ml").is_dir(), "เปิด notebook จาก Product/notebook/"
sys.path.insert(0, str(NB_DIR))
from xau_ml import config as C  # noqa: E402
from xau_ml import cv as CV  # noqa: E402
from xau_ml import data as D  # noqa: E402
from xau_ml import features as F  # noqa: E402
from xau_ml import labels as L  # noqa: E402
from xau_ml import sim as S  # noqa: E402
from xau_ml import smc_ind as SI  # noqa: E402  (backend compute: indicators, smc, smc_setup)
from xau_ml import v2 as V  # noqa: E402

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 40, "display.precision", 4)
print("lightgbm", V.lgb_version(), "| pandas", pd.__version__)

KeyboardInterrupt: 

## Plan (ล็อกไว้ก่อนดูผล)

| ส่วน | ค่าที่ใช้ | เหตุผล |
|---|---|---|
| horizon | intraday 15m, stop 1.5 ATR, time stop 16 แท่ง | เหมือนโมเดล production `xau_intraday_ct25` |
| label | `ct25`: TP 1.5R ตามเทรนด์ daily / 2.5R สวนเทรนด์ | label set ของ production |
| ข้อมูล | broker 15m 2025-01..2026-01, warm-up จาก 2024-10 | ตาม memory: เฉพาะ regime 2025+ |
| validation | walk-forward รายเดือน 2025-04..2026-01, purge + embargo, τ/m และ calibrator แบบ nested (เลือกจากเดือนก่อนหน้าเท่านั้น), burn-in 2 เดือน | protocol v0.2 |
| โมเดล | LightGBM คู่ bull/bear + uniqueness weights + feature ระดับราคา (E5 ของ v0.2) | baseline ที่มี feature ครบสุด |
| ต้นทุน | 0.30 USD/oz ต่อรอบ | `config.COST_PRICE` |

Aug–Sep 2026 holdout ถูกใช้ไปแล้วกับ v0.2 จึงไม่แตะในนี้

In [ ]:
HZ = C.HORIZONS["intraday"]
LS = C.LABEL_SETS["ct25"]
BASE_EXP = next(e for e in V.EXPERIMENTS if e.id == "E5")
ARMS = {
    "M0 ML เดิม": [],
    "M1 + MACD/RSI": SI.IND_COLS,
    "M2 + SMC": SI.SMC_COLS,
    "M3 + ทั้งสอง": SI.IND_COLS + SI.SMC_COLS,
}
print(HZ, LS, BASE_EXP.name, sep="\n")

## 1. ข้อมูลและ feature เดิม

In [ ]:
t0 = time.time()
RAW = {tf: D.load(tf, C.WARMUP_START[tf]) for tf in ("15m", "1h", "1d")}
base = F.build(RAW["15m"], "15m", {tf: RAW[tf] for tf in HZ.htf})
FR = L.add_labels(F.add_extras(base, RAW["1d"]), HZ, LS)
COLS0 = V.columns(BASE_EXP, F.feature_columns(FR))  # baseline columns, fixed BEFORE adding new ones
st = FR.query("date >= @C.TRAIN[0]")
print(f"{len(FR):,} bars {FR['date'].min()} → {FR['date'].max()} | baseline features {len(COLS0)} | "
      f"usable rows {int(CV.usable(st).sum()):,} | long wins {st['y_bull'].mean():.3f} short wins {st['y_bear'].mean():.3f} "
      f"| {time.time() - t0:.0f}s")

## 2. Feature ใหม่ (คำนวณด้วยโค้ด backend ตัวเดียวกับ desk)

`xau_ml/smc_ind.py` เรียก `compute/indicators.py`, `compute/smc.py` (`smc_feature_series` คือ SMC สถานะ ณ ปิดแท่งทุกแท่งใน pass เดียว)
และกฎ `compute/smc_setup.py` จึงไม่มีสูตรที่สอง

In [ ]:
t0 = time.time()
main = RAW["15m"].sort_values("date").reset_index(drop=True)
atr_main = F.base_features(main)["atr14"].to_numpy()
NEW = pd.concat([main[["date"]], SI.indicator_features(main, atr_main), SI.smc_features(main)], axis=1)
NEW["mom_side"] = SI.momentum_side(NEW)
NEW["setup_side"] = SI.setup_side(NEW, main)
FR = FR.merge(NEW, on="date", how="left", validate="one_to_one")
assert len(FR) == len(base)
print(f"new features in {time.time() - t0:.0f}s")
display(FR.loc[FR["date"] >= C.TRAIN[0], SI.IND_COLS + SI.SMC_COLS].describe().T[["count", "mean", "std", "min", "max"]])
sc = FR["date"] >= C.TRAIN[0]
print("momentum rule: long", f"{(FR.loc[sc, 'mom_side'] == 1).mean():.1%}", "short", f"{(FR.loc[sc, 'mom_side'] == -1).mean():.1%}",
      "| setup active bars:", int((FR.loc[sc, 'setup_side'] != 0).sum()), "of", int(sc.sum()))

### ตรวจ look-ahead ด้วยการตัดข้อมูล

คำนวณ feature ใหม่ซ้ำจากข้อมูลที่ตัดถึงแท่งตัดสินใจ แล้วต้องได้ค่าเท่าเดิม (SMC ใช้เวลาต่อรอบ จึงสุ่ม 6 แถว; backend มี test เดียวกันอีกชั้น)

In [ ]:
rng = np.random.default_rng(SEED)
cols_new = SI.IND_COLS + SI.SMC_COLS
worst = 0.0
for t in rng.choice(np.flatnonzero((main["date"] >= C.TRAIN[0]).to_numpy()), 6, replace=False):
    cut = main.iloc[: t + 1]
    again = pd.concat([SI.indicator_features(cut, atr_main[: t + 1]), SI.smc_features(cut)], axis=1).iloc[-1]
    a = NEW.loc[t, cols_new].to_numpy(float)
    b = again[cols_new].to_numpy(float)
    assert np.array_equal(np.isnan(a), np.isnan(b)), main["date"].iloc[t]
    worst = max(worst, float(np.nanmax(np.abs(a - b))))
assert worst < 1e-9, worst
print(f"truncation test on 6 rows: max |diff| = {worst:.1e} ✓")

## 3. Walk-forward folds

In [ ]:
FOLDS = CV.monthly_folds(FR, HZ, C.WF2_FIRST_FOLD, C.WF2_LAST_FOLD, C.TRAIN[0])
CV.assert_folds(FR, FOLDS, HZ)
NAMES = [f.name for f in FOLDS]
display(pd.DataFrame([{"fold": f.name, "train_rows": len(f.train), "test_rows": len(f.test)} for f in FOLDS]).set_index("fold").T)

## 4. ผล ML: AUC ต่อ fold (out-of-fold)

In [ ]:
OOF, PER_FOLD, rows = {}, {}, []
for arm, extra in ARMS.items():
    t0 = time.time()
    cols = COLS0 + extra
    oof, per = V.run_oof(BASE_EXP, FR, FOLDS, cols, HZ)
    OOF[arm], PER_FOLD[arm] = oof, per
    rows.append({"arm": arm, "features": len(cols), "auc_bull": per["auc_bull"].mean(), "auc_bear": per["auc_bear"].mean(),
                 "auc_mean": per[["auc_bull", "auc_bear"]].mean(axis=1).mean(),
                 "logloss_gain_bull": (per["logloss_clim_bull"] - per["logloss_bull"]).mean(),
                 "logloss_gain_bear": (per["logloss_clim_bear"] - per["logloss_bear"]).mean(),
                 "minutes": round((time.time() - t0) / 60, 1)})
ML_TABLE = pd.DataFrame(rows).set_index("arm")
display(ML_TABLE)
auc = pd.DataFrame({arm: per[["auc_bull", "auc_bear"]].mean(axis=1).to_numpy() for arm, per in PER_FOLD.items()}, index=NAMES)
display((auc.sub(auc["M0 ML เดิม"], axis=0)).round(4).rename(columns=lambda c: f"Δ {c}"))

## 5. เทรดแบบ nested out-of-sample (τ/m เลือกจากเดือนก่อนหน้าเท่านั้น)

In [ ]:
TRADES, NEST, rows = {}, {}, []
for arm in ARMS:
    trades, log, cal, sig, scored = V.nested_trading(FR, OOF[arm], NAMES, C.WF2_BURN_IN, C.MIN_VAL_TRADES[HZ.tf])
    TRADES[arm], NEST[arm] = trades, (log, cal, sig, scored)

oof0 = OOF["M0 ML เดิม"]
_, _, SIG0, SCORED0 = NEST["M0 ML เดิม"]
POS = oof0["pos"].to_numpy()[SCORED0]
S0 = SIG0[SCORED0]
mom = FR["mom_side"].to_numpy()[POS]
trend = FR["smc_swing_trend"].to_numpy()[POS]
setup = FR["setup_side"].to_numpy()[POS]

FILTERS = {
    "F1 ML + กรอง RSI/MACD": np.where(mom == S0, S0, 0),
    "F2 ML + ตาม SMC swing trend": np.where(trend == S0, S0, 0),
    "F3 ML + ทั้งสอง": np.where((mom == S0) & (trend == S0), S0, 0),
    "F4 ML + setup เดียวกัน": np.where(setup == S0, S0, 0),
    "R1 setup อย่างเดียว (ไม่มี ML)": setup,
    "R2 momentum อย่างเดียว (ไม่มี ML)": mom,
}
for name, s in FILTERS.items():
    TRADES[name] = S.run(FR, POS, s.astype("int8"))
for kind in ("always_long", "always_short"):
    TRADES[kind] = V.baseline_trades(FR, POS, kind)


def table(names):
    out = []
    base_r = TRADES["M0 ML เดิม"]["r"].to_numpy()
    for n in names:
        t = TRADES[n]
        r = t["r"].to_numpy()
        lo, hi = CV.bootstrap_mean_ci(r) if len(r) > 1 else (np.nan, np.nan)
        dlo, dhi = CV.bootstrap_diff_ci(r, base_r) if len(r) > 1 and n != "M0 ML เดิม" else (np.nan, np.nan)
        out.append({"strategy": n, **S.metrics(t), "R_ci90": f"[{lo:+.3f}, {hi:+.3f}]",
                    "ΔR_vs_M0_ci90": "" if np.isnan(dlo) else f"[{dlo:+.3f}, {dhi:+.3f}]"})
    cols = ["strategy", "trades", "win_rate", "profit_factor", "expectancy_r", "R_ci90", "ΔR_vs_M0_ci90", "total_r", "max_dd_pct", "long_share"]
    return pd.DataFrame(out)[cols].set_index("strategy")


ALL = [*ARMS, *FILTERS, "always_long", "always_short"]
RESULT = table(ALL)
display(RESULT)

## 6. แยกช่วง: ไม่เคยเห็น (2025-06..09) กับที่ v0.1 เคยดู (2025-10..2026-01)

In [ ]:
rows = []
for n in ALL:
    t = TRADES[n]
    if t.empty:
        continue
    seen = (pd.to_datetime(t["entry"]) >= C.SEEN_FROM).to_numpy()
    for part, m in (("never seen", ~seen), ("seen v0.1", seen)):
        tt = t[m].copy()
        tt["equity"] = np.cumprod(1 + C.RISK_PCT * tt["r"].to_numpy())
        mt = S.metrics(tt)
        rows.append({"strategy": n, "period": part, "trades": mt["trades"], "win_rate": mt["win_rate"],
                     "expectancy_r": mt["expectancy_r"], "total_r": mt["total_r"]})
SPLIT = pd.DataFrame(rows).pivot(index="strategy", columns="period", values=["trades", "expectancy_r", "total_r"]).loc[ALL]
display(SPLIT)

## 7. รายเดือน (R รวม)

In [ ]:
monthly = pd.DataFrame({n: TRADES[n].groupby(pd.to_datetime(TRADES[n]["entry"]).dt.to_period("M"))["r"].sum()
                        for n in ALL if not TRADES[n].empty}).round(1)
display(monthly.T)

## 8. เกณฑ์ที่ตั้งไว้: ใครผ่าน

In [ ]:
def verdict(n):
    if n == "M0 ML เดิม" or n.startswith("always"):
        return None
    t = TRADES[n]
    if len(t) < 100:
        return "ไม่ผ่าน: ไม้ < 100"
    dlo, dhi = CV.bootstrap_diff_ci(t["r"].to_numpy(), TRADES["M0 ML เดิม"]["r"].to_numpy())
    sp = SPLIT.loc[n, "expectancy_r"]
    sp0 = SPLIT.loc["M0 ML เดิม", "expectancy_r"]
    worse = [p for p in sp.index if sp[p] < sp0[p]]
    if dlo <= 0:
        return f"ไม่ผ่าน: CI ของผลต่าง [{dlo:+.3f}, {dhi:+.3f}] คร่อม/ต่ำกว่าศูนย์" + (f"; แย่ลงใน {', '.join(worse)}" if worse else "")
    if worse:
        return f"ไม่ผ่าน: แย่ลงใน {', '.join(worse)}"
    return "ผ่าน"


VERDICT = pd.Series({n: verdict(n) for n in ALL}).dropna()
display(VERDICT.to_frame("verdict"))

## 9. โมเดล M3 ให้น้ำหนัก feature ใหม่แค่ไหน (gain, ฝึกบนทุกแถวที่ใช้ได้)

In [ ]:
rows_all = np.flatnonzero(CV.usable(FR) & (FR["date"] >= C.TRAIN[0]).to_numpy())
cols3 = COLS0 + SI.IND_COLS + SI.SMC_COLS
boosters, _ = V.fit_model(BASE_EXP, FR, rows_all, cols3, HZ, C.ModelParams())
imp = pd.DataFrame({s: pd.Series(b.feature_importance("gain"), index=cols3) for s, b in boosters.items()})
imp = imp / imp.sum()
imp["mean"] = imp.mean(axis=1)
imp["group"] = ["MACD/RSI ใหม่" if c in SI.IND_COLS else "SMC" if c in SI.SMC_COLS else "เดิม" for c in imp.index]
display(imp.groupby("group")["mean"].sum().to_frame("share of gain"))
display(imp.sort_values("mean", ascending=False).head(20))

## 10. setup ของ desk ด้วย exit แบบ desk จริง

ข้อ 5 ใช้ exit ของ label ML (stop 1.5 ATR) จึงยังไม่ใช่สิ่งที่ desk เทรด เซลล์นี้จำลองตามกฎ desk:
เข้าที่ open แท่งถัดไป ±5 bps, stop = ไกลกว่าระหว่าง high/low ของแท่ง sweep กับ 2 ATR, TP1 1.6R ปิดครึ่ง,
ที่เหลือถือด้วย stop เดิมถึงแท่งที่ 16, stop ก่อนถ้าโดนทั้งคู่ในแท่งเดียว, ถือทีละไม้ ทุกแท่งตั้งแต่ 2025-01

In [ ]:
SLIP, MIN_ATR, T1, HOLD = 0.0005, 2.0, 1.6, 16
o, h, lo, c = (main[k].to_numpy() for k in ("open", "high", "low", "close"))
side_all = NEW["setup_side"].to_numpy()
since_all = NEW["smc_cont_sweep_since"].to_numpy()
desk, busy = [], -1
for t in range(int(np.flatnonzero((main["date"] >= C.TRAIN[0]).to_numpy())[0]), len(main) - HOLD - 2):
    s = int(side_all[t])
    if s == 0 or t <= busy:
        continue
    j = t - int(since_all[t])
    fill = o[t + 1] * (1 + SLIP * s)
    risk = max(abs(fill - (lo[j] if s > 0 else h[j])), MIN_ATR * atr_main[t])
    stop, tp = fill - s * risk, fill + s * T1 * risk
    half, total, ex = False, 0.0, None
    for k in range(t + 1, t + 1 + HOLD):
        if (s > 0 and lo[k] <= stop) or (s < 0 and h[k] >= stop):
            total += s * (stop * (1 - SLIP * s) - fill) / risk * (0.5 if half else 1.0)
            ex = k
            break
        if not half and ((s > 0 and h[k] >= tp) or (s < 0 and lo[k] <= tp)):
            half, total = True, total + 0.5 * T1
    if ex is None:
        ex = t + HOLD
        total += s * (c[ex] * (1 - SLIP * s) - fill) / risk * (0.5 if half else 1.0)
    busy = ex
    desk.append({"date": main["date"].iat[t], "side": s, "r": total})
DESK = pd.DataFrame(desk)
r = DESK["r"].to_numpy()
lo90, hi90 = CV.bootstrap_mean_ci(r)
print(f"trades {len(r)} | win {np.mean(r > 0):.1%} | R/trade {r.mean():+.3f} CI90 [{lo90:+.3f}, {hi90:+.3f}] | "
      f"total {r.sum():+.1f}R | PF {r[r > 0].sum() / -r[r < 0].sum():.2f} | long {np.mean(DESK['side'] > 0):.0%}")
display(DESK.groupby(DESK["date"].dt.to_period("Q"))["r"].agg(trades="size", mean_r="mean", total_r="sum").round(2))

## Results (รัน 2026-10-05, walk-forward 2025-06..2026-01 ให้คะแนนหลัง burn-in)

**ไม่มีแบบไหนผ่านเกณฑ์ที่ตั้งไว้** ไม่ว่าจะใช้อินดิเคเตอร์ใหม่เป็น feature หรือเป็นตัวกรอง

| | ไม้ | win | R/ไม้ | R รวม | ผลต่างจาก ML เดิม (CI90) |
|---|---|---|---|---|---|
| M0 ML เดิม | 141 | 47.5% | +0.072 | +10.2 | — |
| M1 + MACD/RSI | 263 | 45.6% | +0.018 | +4.8 | [−0.25, +0.14] |
| M2 + SMC | 170 | 45.3% | +0.046 | +7.7 | [−0.23, +0.19] |
| M3 + ทั้งสอง | 89 | 42.7% | −0.028 | −2.5 | [−0.35, +0.16] |
| F1 ML + กรอง RSI/MACD | 103 | 45.6% | +0.048 | +4.9 | [−0.26, +0.21] |
| F2 ML + ตาม SMC trend | 100 | 47.0% | +0.055 | +5.4 | [−0.26, +0.23] |
| R1 setup sweep-continuation อย่างเดียว | 55 | 34.5% | **−0.194** | −10.6 | [−0.55, +0.04] |

- **เป็น feature:** MACD/RSI ใหม่ยกค่า AUC เฉลี่ยขึ้นเล็กน้อย (0.577 → 0.583, ดีขึ้น 8 จาก 10 fold โดยฝั่ง bear ได้มากกว่า) แต่การเทรดแย่ลง โมเดลเปิดไม้เกือบสองเท่าแต่ R ต่อไม้ลดลง ส่วน SMC ไม่ยก AUC แม้จะได้ 13% ของ gain ในโมเดล M3 (MACD/RSI ได้ 2.6%) เพราะโมเดลใช้ SMC แบ่งข้อมูลมากแต่ไม่ได้ทำนายดีขึ้น
- **ช่วงเวลาไม่สอดคล้อง:** M1/M2/M3 ติดลบในช่วงที่ไม่เคยเห็น (2025-06..09) และบวกในช่วงที่ v0.1 เคยดู (2025-10..2026-01) รูปแบบนี้คือ noise ไม่ใช่ edge
- **เป็นตัวกรอง:** ตัดไม้ไป 27–46% แต่ R ต่อไม้ไม่ดีขึ้น ตัวกรองไม่ได้แยกไม้แพ้ออก
- **setup ที่ desk LLM ใช้ (R1) ขาดทุนบน 15m ช่วงนี้** −0.19R ต่อไม้ (55 ไม้ใน 8 เดือน) ต่างจาก rule study บน 1H ที่ได้ +0.23R ซึ่งก็บวกเฉพาะปี 2026 ข้อควรระวัง: ที่นี่ใช้ exit ของ label ML (stop 1.5 ATR, TP 1.5R/2.5R) ไม่ใช่ของ desk (stop ≥ 2 ATR, TP1 1.6R ปิดครึ่ง)
- **ทดสอบซ้ำด้วย exit แบบ desk จริง (ข้อ 10) ยิ่งชัด:** 71 ไม้ win 40.8% **−0.27R ต่อไม้ CI90 [−0.47, −0.06]** PF 0.57 ติดลบ 4 จาก 5 ไตรมาส แปลว่า setup sweep-continuation ที่ใส่ใน desk LLM **ขาดทุนอย่างมีนัยสำคัญ** บนข้อมูล broker 15m ปี 2025 ผลบวกจาก rule study 1H มาจากปี 2026 ปีเดียว
- setup แทบไม่ซ้อนกับสัญญาณ ML (F4 มีไม้เดียว) สองระบบนี้มองคนละจังหวะ
- ML เดิมเปิด long 100% ในช่วงนี้ (ตามที่ ML_RESULTS บันทึกไว้ว่า training เป็น UP 87%)

**ข้อสรุป:** ไม่ควรเอา MACD/RSI/SMC ใหม่เข้า ML ตอนนี้ และ setup ที่ใส่ใน desk LLM ไม่ควรเปิดเป็นค่าเริ่มต้นต่อ เพราะขาดทุนอย่างมีนัยสำคัญบนข้อมูลปี 2025

## Next steps

- ปิด gate (`policy.require_smc_setup = false`) แล้ว desk จะกลับไปใช้ trader v8 ตัดสินใจว่าจะใช้ไหมเป็นของผู้ใช้
- ไม่ต้องเทรน ML รุ่นใหม่ด้วย feature ชุดนี้